In [1]:
# Phase 9: QLoRA fine-tune of Qwen3-1.7B for CTI extraction. Written as percent-cells; `make kaggle-notebook` turns it
# into train_qlora.ipynb. NOT RUN YET (needs a Kaggle GPU): start with DEBUG = True.

# RedString-CTI extractor: QLoRA on Qwen3-1.7B
Kaggle settings: Accelerator GPU T4 x2 (or P100), Internet ON, attach the private dataset `redstring-cti-finetune`,
add Kaggle Secret `HF_TOKEN` (write token). Run with DEBUG = True first, then DEBUG = False via Save & Run All (Commit).

In [2]:
DEBUG = True  # True: 200 examples, 50 steps. False: the full run.
HF_REPO = "abhibarnwal707/redstring-cti-extractor-qwen3-1.7b-lora"
DATA_DIR = "/kaggle/input/datasets/abhibarnwal707/redstring-cti-finetune"
EPOCHS, LR, RANK, SEED = 2, 2e-4, 16, 42

In [3]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 66.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 62.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 11.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 12.6 MB/s eta 0:00:00


In [4]:
from unsloth import FastLanguageModel  # unsloth must be imported before trl / transformers
from unsloth.chat_templates import train_on_responses_only

import json

import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

load_kw = dict(max_seq_length=2048, load_in_4bit=True)
try:
    model, tok = FastLanguageModel.from_pretrained("unsloth/Qwen3-1.7B-unsloth-bnb-4bit", dtype=torch.float16, **load_kw)
except TypeError:  # older unsloth versions name it differently
    model, tok = FastLanguageModel.from_pretrained("unsloth/Qwen3-1.7B-unsloth-bnb-4bit", torch_dtype=torch.float16, **load_kw)
model = FastLanguageModel.get_peft_model(
    model, r=RANK, lora_alpha=2 * RANK, lora_dropout=0, random_state=SEED,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)
# fp16 training needs the trainable (LoRA) weights in float32
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
print({str(p.dtype) for p in model.parameters() if p.requires_grad})  # should print {'torch.float32'}

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.7: Fast Qwen3 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Unsloth 2026.10.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


{'torch.float32'}


## Data
The prompt is rendered exactly as it will be at inference (`add_generation_prompt=True, enable_thinking=False`, which
ends in an empty think block). The target and `<|im_end|>` are appended, so training and inference see the same prefix.

In [5]:
RESPONSE_PART = "<|im_start|>assistant\n<think>\n\n</think>\n\n"


def prompt_of(messages):
    return tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)


def to_text(ex):
    return {"text": prompt_of(ex["messages"]) + ex["messages"][2]["content"] + "<|im_end|>\n"}


raw = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl", "validation": f"{DATA_DIR}/val.jsonl"})
train_raw, val_raw = raw["train"], raw["validation"]
if DEBUG:
    train_raw, val_raw = train_raw.shuffle(seed=SEED).select(range(200)), val_raw.select(range(40))
train_ds, val_ds = train_raw.map(to_text), val_raw.map(to_text)
assert RESPONSE_PART in train_ds[0]["text"], "response marker not found: check the chat template"
print(len(train_ds), len(val_ds)); print(train_ds[0]["text"][-400:])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

200 40
 text must be copied exactly from the input. Use empty lists when nothing applies.<|im_end|>
<|im_start|>user
APT32 has conducted targeted surveillance against activists and bloggers.<|im_end|>
<|im_start|>assistant
<think>

</think>

{"entities":[{"text":"APT32","type":"THREAT_ACTOR"}],"techniques":[{"attack_id":"T1589"}],"relations":[{"head":"APT32","relation":"USES","tail":"T1589"}]}<|im_end|>



In [6]:
import inspect

cfg = SFTConfig(
    dataset_text_field="text", max_length=2048, per_device_train_batch_size=4, gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS, max_steps=50 if DEBUG else -1, learning_rate=LR, warmup_steps=5 if DEBUG else 15,
    fp16=True, bf16=False,  # T4 / P100 have no bf16
    logging_steps=5 if DEBUG else 20, eval_strategy="steps", eval_steps=25 if DEBUG else 100,
    save_steps=25 if DEBUG else 100, save_total_limit=2, output_dir="/kaggle/working/ckpt",
    report_to="none", seed=SEED,
)
# recent trl calls this `processing_class`, older versions `tokenizer`
tok_kw = "processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters else "tokenizer"
trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=cfg, **{tok_kw: tok})
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part=RESPONSE_PART)
trainer.train()  # after a timeout: trainer.train(resume_from_checkpoint=True)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/40 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

...

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
25,0.175080,0.147440
50,0.138120,0.140531


Filter:   0%|          | 0/40 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen3ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/ckpt/checkpoint-25/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/ckpt/checkpoint-50/tokenizer_config.json.


TrainOutput(global_step=50, training_loss=0.24183581709861757, metrics={'train_runtime': 141.1885, 'train_samples_per_second': 5.666, 'train_steps_per_second': 0.354, 'total_flos': 1427273530472448.0, 'train_loss': 0.24183581709861757, 'epoch': 3.88})

## Validation F1 (the number Phase 9 must record)
Greedy decoding on the validation set; JSON validity, entity F1 (exact text + type), technique F1 (ids).

In [7]:
FastLanguageModel.for_inference(model)


def generate(messages):
    ids = tok(prompt_of(messages), return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=700, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def parse(s):
    try:
        d = json.loads(s)
        ents = {(e["text"], e["type"]) for e in d["entities"]}
        techs = {t["attack_id"] for t in d["techniques"]}
        rels = {(r["head"], r["relation"], r["tail"]) for r in d.get("relations") or []}
        return ents, techs, rels
    except Exception:
        return None


def prf(pred, gold):
    tp = len(pred & gold)
    p, r = tp / len(pred) if pred else 0.0, tp / len(gold) if gold else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def micro(pairs):
    tp = sum(len(p & g) for p, g in pairs); np_ = sum(len(p) for p, _ in pairs); ng = sum(len(g) for _, g in pairs)
    p, r = tp / np_ if np_ else 0.0, tp / ng if ng else 0.0
    return round(p, 3), round(r, 3), round(2 * p * r / (p + r), 3) if p + r else 0.0


def evaluate(rows, label):
    valid, ent, tech, rel, bad = 0, [], [], [], []
    for row in rows:
        out = generate(row["messages"])
        gold = parse(row["messages"][2]["content"]); got = parse(out)
        if got is None:
            bad.append(out[:200]); got = (set(), set(), set())
        else:
            valid += 1
        ent.append((got[0], gold[0])); tech.append((got[1], gold[1]))
        if row["task"] == "full":
            rel.append((got[2], gold[2]))
    res = {"split": label, "n": len(rows), "json_valid": round(valid / len(rows), 3),
           "entity_PRF": micro(ent), "technique_PRF": micro(tech), "relation_PRF": micro(rel) if rel else None}
    print(res); [print("INVALID:", b) for b in bad[:3]]
    return res


val_rows = list(val_raw)
results = [evaluate(val_rows, "validation")]
json.dump(results, open("/kaggle/working/validation_metrics.json", "w"), indent=1)

Both `max_new_tokens` (=700) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

{'split': 'validation', 'n': 40, 'json_valid': 1.0, 'entity_PRF': (1.0, 0.833, 0.909), 'technique_PRF': (0.0, 0.0, 0.0), 'relation_PRF': (0.0, 0.0, 0.0)}


## Save: adapter to the Hub (full run only)

In [8]:
if not DEBUG:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    model.push_to_hub(HF_REPO, token=token); tok.push_to_hub(HF_REPO, token=token)
    print("pushed", HF_REPO)
else:
    print("DEBUG run: loss should fall and JSON should mostly parse; nothing was pushed.")

DEBUG run: loss should fall and JSON should mostly parse; nothing was pushed.
